# 14 Karnataka district segments

Question: which Karnataka districts are alike on rain, hospital access and population, so one
programme design could serve a group?

Inputs (all committed): ERA5 district normals (S02, CC BY 4.0), the IMD-based Sen rain trend as an
index (no mm), district hospital distance (D195) and population (S16). Method:
`src/atlas/analysis/segments.py` (robust scaling, k-means for k = 2 to 6 with silhouette, k chosen by a
rule fixed before looking, 1,000 bootstrap resamples scored with the adjusted Rand index, Ward
clustering as a comparison, a run without Bangalore).

With 30 districts this is a typology, not a discovery. `summary.json` records the stability and the
publish decision; see its `limits`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from atlas.analysis.segments import run, write_segments_csv, write_summary

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/14_ka_district_segments"))
OUT.mkdir(parents=True, exist_ok=True)
result = run()
summary = result["summary"]
write_summary(OUT / "summary.json", summary)
write_segments_csv(OUT / "segments.csv", result)
keys = ("k", "silhouette", "bootstrap", "publish_decision")
print(json.dumps({k: summary[k] for k in keys}, indent=1))

In [ ]:
labels, districts = result["labels"], result["districts"]
names = {s["segment"]: s["name"] for s in result["segs"]}
xy = PCA(n_components=2, random_state=0).fit_transform(result["scaled"])
fig, ax = plt.subplots(figsize=(7, 5.5))
for s in sorted(names):
    m = labels == s - 1
    ax.scatter(xy[m, 0], xy[m, 1], label=f"{s}: {names[s]} ({m.sum()})", s=40)
for (x, y), d in zip(xy, districts, strict=True):
    ax.annotate(d, (x, y), fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("principal component 1")
ax.set_ylabel("principal component 2")
ax.set_title("Karnataka districts by segment (robust-scaled features, PCA picture)")
ax.legend(fontsize=7)
fig.savefig(OUT / "segments_pca.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.6))
rows = summary["silhouette_by_k"]
ok = [r["smallest_segment"] >= 3 for r in rows]
a.bar(
    [r["k"] for r in rows],
    [r["silhouette"] for r in rows],
    color=["#2b6cb0" if o else "#bbbbbb" for o in ok],
)
a.set_xlabel("k (grey: a segment has fewer than 3 districts)")
a.set_ylabel("silhouette")
a.set_title(f"Choice of k: k = {summary['k']}")
b.hist(result["ari"], bins=20, color="#2b6cb0")
b.axvline(summary["bootstrap"]["bar"], color="#b3261e", ls="--", label="bar 0.6")
b.set_xlabel("adjusted Rand index vs full-data segments")
b.set_title(f"Bootstrap stability, median {summary['bootstrap']['median_ari']}")
b.legend()
fig.tight_layout()
fig.savefig(OUT / "k_and_stability.png", dpi=80, metadata={"Software": None})
plt.close(fig)
print(summary["segments"])

## Limits

See `summary.json` field `limits`. The segments describe differences in six numbers; they are not
causes, and the names are read off each segment's centroid against the state middle.